In [54]:
# First you need to install the required libraries:
import re
import random
from datasets import load_dataset, Dataset, DatasetDict

random.seed(42)

# ARC DATASET

As  ARC is the first dataset that we used, we'll make them the base format to the others

In [55]:
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")

In [56]:
arc

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})

In [57]:
# proportion analysis
total_questions = len(arc['train']) + len(arc['validation']) + len(arc['test'])
print(f"Total Train Questions: {len(arc['train'])}, Proportion: {len(arc['train'])/total_questions:.2f}")
print(f"Total Validation Questions: {len(arc['validation'])}, Proportion: {len(arc['validation'])/total_questions:.2f}")
print(f"Total Test Questions: {len(arc['test'])}, Proportion: {len(arc['test'])/total_questions:.2f}")

Total Train Questions: 1119, Proportion: 0.43
Total Validation Questions: 299, Proportion: 0.12
Total Test Questions: 1172, Proportion: 0.45


In [58]:
arc['train'][0]

{'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?',
 'choices': {'text': ['dry palms',
   'wet palms',
   'palms covered with oil',
   'palms covered with lotion'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'A'}

# GSMK8

In [59]:
gsm8k      = load_dataset("gsm8k", "main")

In [60]:
gsm8k

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [61]:
def convert_gsm8k_to_arc(split_dict):
    """Convert GSM8K to ARC MCQ format using numeric distractors."""
    # Build pool of all numeric answers for distractor generation
    answer_pool = list({
        m.group(1).strip()
        for split in split_dict.values()
        for ex in split
        if (m := re.search(r"####\s*(.+)", ex["answer"]))
    })

    def convert_split(split, split_name):
        rows = []
        for idx, ex in enumerate(split):
            m = re.search(r"####\s*(.+)", ex["answer"])
            if not m:
                continue
            correct_val = m.group(1).strip()
            distractors = random.sample(
                [a for a in answer_pool if a != correct_val],
                min(3, len(answer_pool) - 1)
            )
            all_opts = [correct_val] + distractors
            random.shuffle(all_opts)
            correct_idx = all_opts.index(correct_val)
            labels = [chr(65 + i) for i in range(len(all_opts))]
            rows.append({
                "id":        f"gsm8k_{split_name}_{idx}",
                "question":  ex["question"],
                "choices":   {"text": all_opts, "label": labels},
                "answerKey": labels[correct_idx],
            })
        return Dataset.from_list(rows)

    return DatasetDict({name: convert_split(split, name) for name, split in split_dict.items()})

In [62]:
gsm8k_arc      = convert_gsm8k_to_arc(gsm8k)
gsm8k_arc

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1319
    })
})

In [63]:
gsm8k_arc['train'][3]

{'id': 'gsm8k_train_3',
 'question': 'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?',
 'choices': {'label': ['A', 'B', 'C', 'D'],
  'text': ['210,000', '9400', '42', '140']},
 'answerKey': 'C'}

# Agieval

In [64]:
agieval_lr = load_dataset("RUCAIBox/agieval", "lsat-lr")
agieval_ar = load_dataset("RUCAIBox/agieval", "lsat-ar")

In [65]:
def convert_agieval_to_arc(split_dict, dataset_name):
    """Convert AGIEval (LSAT-LR / LSAT-AR) to ARC format."""

    def strip_prefix(opt):
        # Remove leading "(A) ", "(B) " etc.
        return re.sub(r"^\([A-Z]\)\s*", "", opt).strip()

    def convert_split(split, split_name):
        rows = []
        for idx, ex in enumerate(split):
            passage  = (ex.get("passage") or "").strip()
            question = ex["question"].strip()
            full_q   = f"{passage}\n\n{question}" if passage else question
            options  = ex["options"]
            texts    = [strip_prefix(o) for o in options]
            labels   = [chr(65 + i) for i in range(len(options))]
            label    = ex["label"]
            if isinstance(label, (int, float)) or str(label).isdigit():
                answer_key = chr(65 + int(label))
            else:
                answer_key = str(label).strip().upper()
            rows.append({
                "id":        f"{dataset_name}_{split_name}_{idx}",
                "question":  full_q,
                "choices":   {"text": texts, "label": labels},
                "answerKey": answer_key,
            })
        return Dataset.from_list(rows)

    return DatasetDict({name: convert_split(split, name) for name, split in split_dict.items()})

In [66]:
agieval_lr_arc = convert_agieval_to_arc(agieval_lr, "agieval_lr")
agieval_ar_arc = convert_agieval_to_arc(agieval_ar, "agieval_ar")

In [67]:
agieval_lr_arc

DatasetDict({
    dev: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 3
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 510
    })
})

#  AQuA-RAT

In [68]:
aqua       = load_dataset("aqua_rat")
aqua

DatasetDict({
    train: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 97467
    })
    test: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 254
    })
    validation: Dataset({
        features: ['question', 'options', 'rationale', 'correct'],
        num_rows: 254
    })
})

In [69]:
def convert_aqua_to_arc(split_dict):
    """Convert AQuA-RAT to ARC format."""

    def strip_prefix(opt):
        # Remove leading "A) ", "B) " etc.
        return re.sub(r"^[A-Z]\)\s*", "", opt).strip()

    def convert_split(split, split_name):
        rows = []
        for idx, ex in enumerate(split):
            options = ex["options"]
            texts   = [strip_prefix(o) for o in options]
            labels  = [chr(65 + i) for i in range(len(options))]
            rows.append({
                "id":        f"aqua_{split_name}_{idx}",
                "question":  ex["question"].strip(),
                "choices":   {"text": texts, "label": labels},
                "answerKey": ex["correct"].strip().upper(),
            })
        return Dataset.from_list(rows)

    return DatasetDict({name: convert_split(split, name) for name, split in split_dict.items()})


In [70]:
aqua_arc       = convert_aqua_to_arc(aqua)
aqua_arc

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 97467
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 254
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 254
    })
})

In [71]:
aqua_arc['train'][0]

{'id': 'aqua_train_0',
 'question': "Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?",
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
  'text': ['21', '21.5', '22', '22.5', '23']},
 'answerKey': 'E'}

# MMLU-PRO

In [72]:
mmlu_pro   = load_dataset("TIGER-Lab/MMLU-Pro")

In [73]:
def convert_mmlu_pro_to_arc(split_dict):
    """Convert MMLU-Pro to ARC format."""

    def convert_split(split, split_name):
        rows = []
        for idx, ex in enumerate(split):
            options = ex["options"]
            labels  = [chr(65 + i) for i in range(len(options))]
            rows.append({
                "id":        str(ex.get("question_id", f"mmlu_pro_{split_name}_{idx}")),
                "question":  ex["question"].strip(),
                "choices":   {"text": options, "label": labels},
                "answerKey": ex["answer"].strip().upper(),
            })
        return Dataset.from_list(rows)

    return DatasetDict({name: convert_split(split, name) for name, split in split_dict.items()})

mmlu_pro_arc   = convert_mmlu_pro_to_arc(mmlu_pro)

In [74]:
mmlu_pro_arc

DatasetDict({
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 12032
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 70
    })
})

# Salvando os Datasets

In [75]:
import os

datasets_to_save = {
    "gsm8k":      gsm8k_arc,
    "agieval_lr": agieval_lr_arc,
    "agieval_ar": agieval_ar_arc,
    "aqua":       aqua_arc,
    "mmlu_pro":   mmlu_pro_arc,
}

for name, ds_dict in datasets_to_save.items():
    folder = os.path.join("datasets", name)
    os.makedirs(folder, exist_ok=True)
    for split_name, split_data in ds_dict.items():
        filepath = os.path.join(folder, f"{split_name}.parquet")
        split_data.to_parquet(filepath)
        print(f"Saved: {filepath} ({len(split_data)} records)")

print("\nAll datasets saved to datasets/ in ARC format!")
print("\nFolder structure:")
for root, dirs, files in os.walk("datasets"):
    level = root.replace("datasets", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".parquet"):
            print(f"{sub}{file}")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 54.09ba/s]


Saved: datasets\gsm8k\train.parquet (7473 records)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 150.01ba/s]


Saved: datasets\gsm8k\test.parquet (1319 records)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1142.86ba/s]


...

Saved: datasets\aqua\validation.parquet (254 records)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 21.35ba/s]


Saved: datasets\mmlu_pro\test.parquet (12032 records)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 371.54ba/s]

Saved: datasets\mmlu_pro\validation.parquet (70 records)

All datasets saved to datasets/ in ARC format!

Folder structure:
datasets/
  agieval_ar_dev.parquet
  agieval_ar_test.parquet
  agieval_lr_dev.parquet
  agieval_lr_test.parquet
  aqua_test.parquet
  aqua_validation.parquet
  arc_challenge_test.parquet
  arc_challenge_train.parquet
  arc_challenge_validation.parquet
  gsm8k_test.parquet
  gsm8k_train.parquet
  mmlu_pro_validation.parquet
  agieval_ar/
    dev.parquet
    test.parquet
  agieval_lr/
    dev.parquet
    test.parquet
  aqua/
    test.parquet
    train.parquet
    validation.parquet
  gsm8k/
    test.parquet
    train.parquet
  mmlu_pro/
    test.parquet
    validation.parquet


In [43]:
# Verificar um exemplo do dataset
print("Splits disponíveis:", list(ds.keys()))
print("\n--- Exemplo do split 'train' ---")
example = ds['train'][0]
for key, value in example.items():
    print(f"{key}: {value}")

Splits disponíveis: ['train', 'test', 'validation']

--- Exemplo do split 'train' ---
id: Mercury_SC_415702
question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
choices: {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}
answerKey: A


In [44]:
import os

# Criar pasta datasets
os.makedirs("datasets", exist_ok=True)

# Salvar cada split como Parquet (preserva dicionários e listas)
for split_name, split_data in ds.items():
    filepath = f"datasets/arc_challenge_{split_name}.parquet"
    split_data.to_parquet(filepath)
    print(f"Salvo: {filepath} ({len(split_data)} registros)")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 161.54ba/s]


Salvo: datasets/arc_challenge_train.parquet (1119 registros)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 194.24ba/s]


Salvo: datasets/arc_challenge_test.parquet (1172 registros)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 216.34ba/s]

Salvo: datasets/arc_challenge_validation.parquet (299 registros)


In [45]:
from datasets import Dataset

# Ler um split do Parquet
train = Dataset.from_parquet("datasets/arc_challenge_train.parquet")
print(train)
print("\n--- Primeiro exemplo ---")
print(train[0])

Generating train split: 1119 examples [00:00, 94564.52 examples/s]

Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 1119
})

--- Primeiro exemplo ---
{'id': 'Mercury_SC_415702', 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'choices': {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'A'}


# Trying to answer questions

In [46]:
train[0]

{'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?',
 'choices': {'text': ['dry palms',
   'wet palms',
   'palms covered with oil',
   'palms covered with lotion'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'A'}

In [47]:
! pip install typing

In [57]:
from typing import TypedDict

def format_sample(sample:TypedDict)->str:
    question = sample['question']
    choices = sample['choices']['text']
    answer_key = sample['answerKey']
    correct_answer = choices[ord(answer_key) - ord('A')]
    
    formatted = f"Question: {question}\nChoices:\n"
    for idx, choice in enumerate(choices):
        formatted += f"{chr(ord('A') + idx)}. {choice}\n"
    formatted += f"Answer: {correct_answer}"
    
    return formatted

def get_question(sample:TypedDict)->str:
    question = sample['question']
    return question

def get_alternatives_label(sample:TypedDict)->str:
    label = sample['choices']['label']
    return label

def get_alternatives_text(sample:TypedDict)->str:
    text = sample['choices']['text']
    return text

def get_correct_answer(sample:TypedDict)->str:
    answer_key = sample['answerKey']
    return answer_key

def get_correct_answer_text(sample:TypedDict)->str:
    correct_answer = get_correct_answer(sample)
    labels = get_alternatives_label(sample)
    index = labels.index(correct_answer)
    text = sample['choices']['text'][index]
    return text

def try_to_answer(sample:TypedDict, guess:str)->bool:
    correct_answer = get_correct_answer(sample).strip().upper() 
    treated_guess = guess.strip().upper()
    if correct_answer == treated_guess:
        return True
    else:
        return False
    
def format_question(sample:TypedDict)->str:
    question = get_question(sample)
    texts = get_alternatives_text(sample)
    labels = get_alternatives_label(sample)

    formatted_choices = "\n".join(
        [f"({label}) {choice}" for label, choice in zip(labels, texts)]
    )
    formatted_question = f"{question}\n{formatted_choices}"

    return formatted_question

In [59]:
print(format_question(sample=train[0]))

George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
(A) dry palms
(B) wet palms
(C) palms covered with oil
(D) palms covered with lotion


In [55]:
train[0]

{'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?',
 'choices': {'text': ['dry palms',
   'wet palms',
   'palms covered with oil',
   'palms covered with lotion'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'A'}